# Teaching a network to copy the heuristic

**Behavior cloning**: record what the hand-written heuristic does in thousands of situations, then train a small network to make the same choices. No rewards, no trial and error yet, just "here's the situation, here's what the expert did, learn to do that." It's plain supervised learning, the same idea as teaching a model to label photos.

Work through it a pocket at a time. Each pocket ends somewhere you can stop:

1. **Record the teacher**: done for you, just run it
2. **Meet PyTorch**: a tiny network, one forward pass
3. **The training loop**: make the loss go down
4. **Is it really learning?**: held-out games, accuracy per action
5. **Let the clone fish**: catches on the real game, against the heuristic's ~14
6. **Pick a fish**: the network chooses which fish, code does the aiming

## Pocket 1: record the teacher

Every time the game asks for a decision, we save two things:

- **the input**: `encode(obs)`, your 40 numbers
- **the answer**: what the heuristic did, as a *type* (0 = wait, 1 = reel, 2 = cast) plus, for casts, *where* (x and y scaled to 0..1)

We record games on seeds **100 to 299**, and keep seeds **1 to 20** unseen for the final test in pocket 5. Testing a model on the same games it learned from would be like grading a test using the practice questions.

The result is saved to `training/data/teacher.npz` (NumPy's file format for arrays), so later pockets can just load it, even after a restart. It takes about half a minute.

In [1]:
import time
from collections import Counter
from pathlib import Path

import numpy as np

from fishing_env import FishingEnv
from features import encode, action_to_target, ACTION_TYPES, N_FEATURES

TRAIN_SEEDS = range(100, 300)   # 200 games to learn from
DATA = Path("data/teacher.npz")

X, y_type, y_pos, game = [], [], [], []
t0 = time.perf_counter()
with FishingEnv() as env:
    for i, seed in enumerate(TRAIN_SEEDS):
        obs, info = env.reset(seed=seed)
        done = False
        while not done:
            kind, pos = action_to_target(info["teacher"])
            X.append(encode(obs))
            y_type.append(kind)
            y_pos.append(pos)
            game.append(seed)
            obs, reward, done, _, info = env.step(info["teacher"])
        if (i + 1) % 50 == 0:
            print(f"{i + 1} games, {len(X)} decisions, {time.perf_counter() - t0:.0f}s")

DATA.parent.mkdir(exist_ok=True)
np.savez(
    DATA,
    X=np.array(X, dtype=np.float32),
    y_type=np.array(y_type, dtype=np.int64),
    y_pos=np.array(y_pos, dtype=np.float32),
    game=np.array(game, dtype=np.int64),
)
print("saved", DATA)

50 games, 10640 decisions, 3s


100 games, 21237 decisions, 6s


150 games, 31894 decisions, 9s


200 games, 42455 decisions, 12s
saved data\teacher.npz


Now look at what you recorded. Pay attention to how the three action types are split. It'll matter in pocket 4.

In [2]:
d = np.load(DATA)
X, y_type, y_pos, game = d["X"], d["y_type"], d["y_pos"], d["game"]
print("inputs  X:", X.shape, " (decisions, features)")
print("answers y_type:", y_type.shape, " y_pos:", y_pos.shape)
counts = Counter(y_type.tolist())
for k, name in enumerate(ACTION_TYPES):
    print(f"  {name:5s} {counts[k]:6d}  {100 * counts[k] / len(y_type):5.1f}%")

inputs  X:

 (42455, 40)  (decisions, features)
answers y_type: (42455,)  y_pos: (42455, 2)
  wait   30228   71.2%
  reel    6035   14.2%
  cast    6192   14.6%


**Stop here for pocket 1.** You've got a dataset: roughly 40,000 situations, each with the expert's answer. Notice the split: the heuristic waits most of the time. A model that learned only "always wait" would be right most of the time and never catch a fish. Hold that thought until pocket 4.

---

## Pocket 2: meet PyTorch

A **tensor** is PyTorch's array: like a NumPy array, but it can track how every number was computed, and that's what lets it learn. A **network** is a stack of layers. `nn.Linear(40, 64)` takes 40 numbers in and gives 64 out (a grid of 40 × 64 weights, plus 64 biases). `nn.ReLU()` keeps positive numbers and turns negative ones into 0, which is what lets stacked layers learn more than a straight line.

Ours needs **5 outputs**: 3 scores for the action type (wait, reel, cast) and 2 numbers for where to cast.

In [3]:
import torch
from torch import nn

# TODO: turn the NumPy arrays into tensors
#   torch.from_numpy(X) works, for X, y_type and y_pos
X_t = torch.from_numpy(X)
y_type_t = torch.from_numpy(y_type)
y_pos_t = torch.from_numpy(y_pos)


# TODO: build the network: 40 in, two hidden layers of 64 with ReLU, 5 out
model = nn.Sequential(
  nn.Linear(N_FEATURES, 64), nn.ReLU(),
  nn.Linear(64, 64), nn.ReLU(),
  nn.Linear(64, 5),
)

# TODO: push the first decision through it and print the 5 numbers it gives back.
#   The model wants a batch, so give it X_t[:1] (one row, as a tensor), not X_t[0].
#   They'll be meaningless for now. Nothing is trained yet.

model(X_t[:1])

tensor([[ 0.0397, -0.0314, -0.0943, -0.0885, -0.0099]],
       grad_fn=<AddmmBackward0>)

---

## Pocket 3: the training loop

Training repeats four lines, over and over:

1. **predict**: run a batch of inputs through the model
2. **measure how wrong it was**: the *loss*. Two parts here:
   - the action type is a choice between classes, so it uses `nn.CrossEntropyLoss`
   - the cast position is two numbers, so it uses `nn.MSELoss` (mean squared error). Only count it on decisions where the teacher actually cast, since position means nothing for wait or reel.
3. **work out how to nudge every weight** to be less wrong: `loss.backward()`
4. **nudge them**: `optimizer.step()`, then `optimizer.zero_grad()` to start fresh

One pass over all the data is an **epoch**. Data goes in **batches** (say 256 rows at a time), shuffled each epoch.

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
type_loss_fn = nn.CrossEntropyLoss()
pos_loss_fn = nn.MSELoss()

for epoch in range(10):
    perm = torch.randperm(len(X_t))
    total, batches = 0.0, 0
    for start in range(0, len(X_t), 256):
        idx = perm[start:start + 256]
        out = model(X_t[idx])
        type_scores, pos = out[:, :3], out[:, 3:]

        loss = type_loss_fn(type_scores, y_type_t[idx])
        is_cast = y_type_t[idx] == 2
        if is_cast.any():
            loss = loss + pos_loss_fn(pos[is_cast], y_pos_t[idx][is_cast])

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total += loss.item()
        batches += 1
    print(epoch, total / batches)

0 0.5233668040439307


1 0.2252593139208943


2 0.20205525246011205


3 0.19574515683105193


4 0.189972649721137


5 0.18408337100801697


6 0.17908450624490357


7 0.1738472339439105


8 0.16970021035297808


9 0.1644597403615354


---

## Pocket 4: is it really learning?

A falling loss means the model fits the data it trained on, but that's like memorizing practice questions. The real question is whether it does well on **games it has never seen**.

- **Split by game, not by row**: train on most games, hold out a few whole games (the `game` array says which game each row came from). Rows from the same game are near-duplicates, so a random row split would quietly leak answers.
- **Accuracy per action type**: overall accuracy will look great because most answers are "wait." Check how often it gets **reel** and **cast** right on their own. Those are the ones that catch fish.

In [5]:
# ---- Split by game, not by row ----
# Seeds 100..259 are for learning; 260..299 are held out as games the model never sees.
# `game` says which game each row came from, so this gives one True/False per row.
val = game >= 260
train = ~val                     # ~ flips every True/False

# Turn the masks into tensors so they can slice the tensors from pocket 2
val_t, train_t = torch.from_numpy(val), torch.from_numpy(train)
X_train, X_val = X_t[train_t], X_t[val_t]            # flashcard fronts
yt_train, yt_val = y_type_t[train_t], y_type_t[val_t]  # answer: type
yp_train, yp_val = y_pos_t[train_t], y_pos_t[val_t]    # answer: cast position
print(f"train: {len(X_train)} rows   validation: {len(X_val)} rows")


# The same loss as pocket 3, wrapped in a function so it can measure
# a training batch and the validation games the same way
def compute_loss(model, X, yt, yp):
    out = model(X)
    loss = type_loss_fn(out[:, :3], yt)            # how wrong the wait/reel/cast choice was
    is_cast = yt == 2
    if is_cast.any():                               # position only counts on casts
        loss = loss + pos_loss_fn(out[:, 3:][is_cast], yp[is_cast])
    return loss


# ---- A fresh network ----
# The pocket 3 model already studied the validation games, so it can't be tested on them.
# This one starts from random knobs and only ever sees the training games.
model = nn.Sequential(
    nn.Linear(N_FEATURES, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 5),
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ---- Train, and check both losses after every epoch ----
for epoch in range(10):
    perm = torch.randperm(len(X_train))
    for start in range(0, len(X_train), 256):
        idx = perm[start:start + 256]
        loss = compute_loss(model, X_train[idx], yt_train[idx], yp_train[idx])
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    # no_grad: just measuring, not learning, so PyTorch can skip recording the math
    with torch.no_grad():
        train_loss = compute_loss(model, X_train, yt_train, yp_train).item()
        val_loss = compute_loss(model, X_val, yt_val, yp_val).item()
    print(f"epoch {epoch}   train loss {train_loss:.3f}   validation loss {val_loss:.3f}")

# ---- Accuracy per action type, on the held-out games ----
with torch.no_grad():
    # argmax: which of the 3 type scores is highest -> 0 wait, 1 reel, 2 cast
    pred = model(X_val)[:, :3].argmax(dim=1)

print()
print(f"always wait would get: {(yt_val == 0).float().mean().item():.1%}")
print(f"overall accuracy:      {(pred == yt_val).float().mean().item():.1%}")
for k, name in enumerate(ACTION_TYPES):
    rows = yt_val == k                              # the rows where the teacher chose this type
    right = (pred[rows] == k).float().mean().item()  # how often the model chose it too
    print(f"  {name:5s} {right:6.1%} right, out of {rows.sum().item()} rows")

train: 33997 rows   validation: 8458 rows


epoch 0   train loss 0.278   validation loss 0.290
epoch 1   train loss 0.218   validation loss 0.232


epoch 2   train loss 0.199   validation loss 0.216
epoch 3   train loss 0.194   validation loss 0.211


epoch 4   train loss 0.190   validation loss 0.206
epoch 5   train loss 0.186   validation loss 0.203


epoch 6   train loss 0.184   validation loss 0.200
epoch 7   train loss 0.180   validation loss 0.199


epoch 8   train loss 0.178   validation loss 0.199


epoch 9   train loss 0.170   validation loss 0.191

always wait would get: 71.0%
overall accuracy:      96.0%
  wait   99.3% right, out of 6008 rows
  reel   75.4% right, out of 1211 rows
  cast  100.0% right, out of 1239 rows


---

## Pocket 5: let the clone fish

Now the real test: the clone plays actual games, on seeds 1 to 20 it never saw. `clone_policy` turns an observation into an action. The evaluation loop below is done for you. It reports average catches, to compare against the heuristic's **about 14** on the same seeds.

In [6]:
from features import target_to_action


def clone_policy(obs):
    x = torch.tensor([encode(obs)])     # dict -> 40 numbers, as a batch of one: (1, 40)
    out = model(x)[0]                   # forward pass, take the one row: 5 numbers
    kind = out[:3].argmax().item()      # highest of wait/reel/cast -> 0, 1 or 2
    pos = out[3:].tolist()              # where to cast, scaled 0..1
    return target_to_action(kind, pos)  # back into an action the game understands


def evaluate(policy, seeds=range(1, 21)):
    catches = []
    with FishingEnv() as env:
        for seed in seeds:
            obs, info = env.reset(seed=seed)
            done = False
            while not done:
                with torch.no_grad():      # no learning during play
                    action = policy(obs)
                obs, reward, done, _, info = env.step(action)
            catches.append(info["stats"]["catches"])
    return sum(catches) / len(catches), catches


mean, per_seed = evaluate(clone_policy)
print(f"clone: {mean:.1f} catches per game (heuristic: about 14)")
print(per_seed)

clone: 8.9 catches per game (heuristic: about 14)
[9, 6, 8, 10, 10, 9, 7, 13, 8, 7, 10, 10, 8, 8, 7, 9, 9, 8, 9, 14]


---

## Pocket 6: pick a fish

The first clone matched the heuristic's choices 96% of the time but caught only about 9 fish to its 14. Training longer and weighting the reels didn't help. The split test found why: the clone's choices with the heuristic's aim catch 13.6. **The whole gap was aim.** Its casts missed by about 61px, and fish only notice bait within 30px.

The heuristic picks the best of 30 fish and then leads it. A network that outputs one x,y, trained with MSE, splits the difference when two fish look good, and casts into the empty water between them.

So the job is split in two:

- **the brain (the network)** picks *which* of the 5 fish in your `encode()` to go after. It's a choice, like wait/reel/cast, so it uses CrossEntropy and can't average between fish.
- **the hands (plain code)**: `lead_for` in features.py works out where to drop the hook for that fish. It's the heuristic's own aiming formula, ported line for line.

The output layer goes from 5 numbers to **3 + 5 = 8**: three type scores, then one score per fish.

**6a: label which fish each cast was aimed at.** The game is deterministic, so replaying the same 200 games gives exactly the same rows as pocket 1. This time, each teacher cast also gets a label: which of the 5 nearest fish it was aimed at, or -1 if it went after one farther away.

In [7]:
from features import K, nearest_fish, lead_for, target_slot, cast, WAIT, REEL

slot = []
X_check = []
with FishingEnv() as env:
    for seed in TRAIN_SEEDS:
        obs, info = env.reset(seed=seed)
        done = False
        while not done:
            teacher = info["teacher"]
            X_check.append(encode(obs))
            slot.append(target_slot(obs, teacher) if teacher["type"] == "cast" else -1)
            obs, reward, done, _, info = env.step(teacher)

slot = np.array(slot, dtype=np.int64)
print("same games as pocket 1:", np.array_equal(np.array(X_check, dtype=np.float32), X))
casts = y_type == 2
print(f"casts: {casts.sum()}, aimed at one of the {K} nearest fish: {(slot[casts] >= 0).mean():.1%}")

same games as pocket 1: True
casts: 6192, aimed at one of the 5 nearest fish: 38.6%


**6b: train the picker.** Same network shape and the same game split as pocket 4, but 8 outputs. The "which fish" part only counts on casts that went after one of the 5 fish, since the others have no right answer among them.

In [8]:
torch.manual_seed(0)   # fixed starting knobs, so rerunning gives the same model (and the same shipped weights)

slot_t = torch.from_numpy(slot)
s_train, s_val = slot_t[train_t], slot_t[val_t]   # same game split as pocket 4

# 3 + K outputs: wait/reel/cast scores, then one score per fish
pick_model = nn.Sequential(
    nn.Linear(N_FEATURES, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 3 + K),
)
optimizer = torch.optim.Adam(pick_model.parameters(), lr=1e-3)
slot_loss_fn = nn.CrossEntropyLoss()   # "which fish" is a choice, like the action type

def pick_loss(model, X, yt, s):
    out = model(X)
    loss = type_loss_fn(out[:, :3], yt)
    labeled = s >= 0                   # only casts aimed at one of the 5 fish have a right answer
    if labeled.any():
        loss = loss + slot_loss_fn(out[:, 3:][labeled], s[labeled])
    return loss

for epoch in range(10):
    perm = torch.randperm(len(X_train))
    for start in range(0, len(X_train), 256):
        idx = perm[start:start + 256]
        loss = pick_loss(pick_model, X_train[idx], yt_train[idx], s_train[idx])
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    with torch.no_grad():
        train_loss = pick_loss(pick_model, X_train, yt_train, s_train).item()
        val_loss = pick_loss(pick_model, X_val, yt_val, s_val).item()
    print(f"epoch {epoch}   train loss {train_loss:.3f}   validation loss {val_loss:.3f}")

with torch.no_grad():
    labeled = s_val >= 0
    picked = pick_model(X_val)[:, 3:][labeled].argmax(dim=1)
print(f"picked the teacher's fish on {(picked == s_val[labeled]).float().mean().item():.1%} of held-out casts")

epoch 0   train loss 1.557   validation loss 1.584

epoch 1   train loss 1.010   validation loss 1.056
epoch 2   train loss 0.809   validation loss 0.876


epoch 3   train loss 0.745   validation loss 0.834
epoch 4   train loss 0.722   validation loss 0.814


epoch 5   train loss 0.761   validation loss 0.843
epoch 6   train loss 0.709   validation loss 0.807


epoch 7   train loss 0.755   validation loss 0.878
epoch 8   train loss 0.745   validation loss 0.872


epoch 9   train loss 0.709   validation loss 0.803
picked the teacher's fish on 75.0% of held-out casts


**6c: let it fish.** On a cast, try the fish in order of the network's preference, skip any that are scared, missing, or about to turn at the glass, and let `lead_for` do the aiming.

In [9]:
def pick_policy(obs):
    out = pick_model(torch.tensor([encode(obs)]))[0]
    kind = out[:3].argmax().item()
    if kind == 0:
        return WAIT
    if kind == 1:
        return REEL
    fish = nearest_fish(obs)
    for j in out[3:].argsort(descending=True).tolist():
        if j < len(fish) and not fish[j]["scared"]:
            spot = lead_for(fish[j], obs)
            if spot:
                return cast(*spot)
    return WAIT

mean, per_seed = evaluate(pick_policy)
print(f"pick-a-fish: {mean:.1f} catches per game (heuristic: 13.9, first clone: about 9)")
print(per_seed)

pick-a-fish: 13.7 catches per game (heuristic: 13.9, first clone: about 9)
[14, 7, 10, 12, 12, 15, 11, 14, 13, 16, 16, 13, 12, 18, 11, 15, 13, 20, 15, 17]


---

## Pocket 7: ship it

The trained network is just numbers: a grid of weights and a list of biases for each layer, about 7,300 in all. This writes them to a JSON file the game imports. In the browser, `src/game/hunters/clone.ts` does the same forward pass in plain TypeScript, with no PyTorch, then lets the heuristic's own aiming code drop the hook.

`training/parity.py` checks that both sides agree: the same 40 features, the same choices, and the same catches.

In [10]:
import json

layers = [m for m in pick_model if isinstance(m, nn.Linear)]
exported = {
    "about": "pick-a-fish clone, trained in training/clone.ipynb. layers run in order, ReLU between them.",
    "features": N_FEATURES,
    "K": K,
    "catches": mean,   # on seeds 1-20, from 6c
    "layers": [
        {
            "w": [[float(f"{v:.6g}") for v in row] for row in layer.weight.tolist()],   # (out, in)
            "b": [float(f"{v:.6g}") for v in layer.bias.tolist()],
        }
        for layer in layers
    ],
}
out_path = Path("../src/game/hunters/clone-weights.json")
out_path.write_text(json.dumps(exported, separators=(",", ":")))
print(f"wrote {out_path}: {sum(p.numel() for p in pick_model.parameters())} numbers, {out_path.stat().st_size / 1024:.0f} KB")

wrote ..\src\game\hunters\clone-weights.json: 7304 numbers, 71 KB
